ejercicio 5.1

In [ ]:
import random

def jugar_adivinanza(estadisticas):
    print("\n" + "=" * 40)
    print("      JUEGO DE ADIVINANZA DE NÚMEROS")
    print("=" * 40)
    print("He pensado un número entre 1 y 100.")
    print("Tienes máximo 7 intentos para adivinarlo.")

    numero_secreto = random.randint(1, 100)
    historial_intentos = []
    intentos_restantes = 7
    gano_partida = False

    while intentos_restantes > 0:
        print(f"\nTe quedan {intentos_restantes} intentos.")

        while True:
            entrada_usuario = input("Introduce tu adivinanza: ")

            if not entrada_usuario.isdigit():
                print("Por favor, introduce un número entero válido.")
                continue

            intento = int(entrada_usuario)
            if intento < 1 or intento > 100:
                print("El número debe estar entre 1 y 100.")
                continue

            break

        historial_intentos.append(intento)
        intentos_restantes -= 1

        if intento == numero_secreto:
            gano_partida = True
            break
        elif intento < numero_secreto:
            print("Demasiado bajo. Es mayor.")
        else:
            print("Demasiado alto. Es menor.")

    print("\n" + "-" * 40)
    if gano_partida:
        intentos_usados = 7 - intentos_restantes
        print(f"¡FELICIDADES! Adivinaste el número en {intentos_usados} intentos.")
        estadisticas["ganadas"] += 1
    else:
        print(f"Se acabaron los intentos. El número era {numero_secreto}.")
        estadisticas["perdidas"] += 1

    print(f"Historial de tus adivinanzas: {historial_intentos}")
    print(f"Marcador Actual: Ganadas: {estadisticas['ganadas']}, Perdidas: {estadisticas['perdidas']}")
    print("-" * 40)


marcador_global = {
    "ganadas": 0,
    "perdidas": 0
}

querer_jugar = True

while querer_jugar:
    jugar_adivinanza(marcador_global)

    while True:
        respuesta = input("\n¿Deseas jugar otra vez? (s/n): ").lower().strip()
        if respuesta == 's':
            break
        elif respuesta == 'n':
            querer_jugar = False
            break
        else:
            print("Opción no válida. Ingresa 's' para sí o 'n' para no.")

### Explicación: ¿Por qué 7 intentos son suficientes? (Relación con Búsqueda Binaria)

* **Estrategia óptima (División y Conquista):**
  La forma óptima de jugar es seleccionar siempre el número del centro del espacio de búsqueda restante. Si el rango actual es $[1, 100]$, la primera adivinanza debe ser $50$.
  * Si el programa indica *"mayor"*, el espacio de búsqueda se reduce a la mitad superior: $[51, 100]$.
  * Si indica *"menor"*, el espacio se reduce a la mitad inferior: $[1, 49]$.

---

* **Análisis Matemático ($O(\log_2 n)$):**
  Cada intento reduce el espacio de opciones posibles exactamente a la mitad ($\frac{N}{2}$). Este comportamiento sigue una progresión logarítmica en base 2:

  $$\text{Intentos máximos} = \lceil \log_2(N) \rceil$$

  Para un espacio de $N = 100$ elementos:
  $$\log_2(100) \approx 6.64$$

  Redondeando hacia el entero superior, se concluye que **se requieren máximo 7 intentos** para garantizar encontrar cualquier número entre 1 y 100.

---

* **Reducción paso a paso del espacio de búsqueda en el peor de los casos:**

| Intento | Opciones restantes ($N$) | Selección óptima (Punto medio) |
| :---: | :---: | :---: |
| **1** | 100 | $50$ |
| **2** | 50 | $25$ o $75$ |
| **3** | 25 | $12$, $37$, $62$ u $87$ |
| **4** | 12 | Reducción a ~6 elementos |
| **5** | 6 | Reducción a ~3 elementos |
| **6** | 3 | Reducción a ~1 elemento |
| **7** | 1 | **Número encontrado con certeza** |

---

* **Conclusión:**
  Como $2^7 = 128$ (capacidad de cobertura de 7 decisiones binarias), 7 intentos son matemáticamente más que suficientes para cubrir un rango de 100 posibilidades sin importar qué número elija el sistema.

ejercicio 5.2

In [ ]:
import time


ANCHO = 10
ALTO = 10
GENERACIONES_A_MOSTRAR = 5

tablero_actual = []
for f in range(ALTO):
    fila = []
    for c in range(ANCHO):
        fila.append(0)
    tablero_actual.append(fila)

tablero_actual[1][2] = 1
tablero_actual[2][3] = 1
tablero_actual[3][1] = 1
tablero_actual[3][2] = 1
tablero_actual[3][3] = 1



def mostrar_tablero(tablero, gen):
    print(f"\n--- Generación {gen} ---")
    header = "   "
    for c in range(ANCHO): header += f"{c} "
    print(header)
    print("  " + "-" * (ANCHO * 2 + 1))

    for f in range(ALTO):
        linea = f"{f} | "
        for c in range(ANCHO):
            celda = tablero[f][c]
            if celda == 1:
                linea += "■ "
            else:
                linea += "· "
        print(linea)
    print("  " + "-" * (ANCHO * 2 + 1))


def contar_vecinos_vivos(tablero, f, c):
    total = 0
    for df in range(-1, 2):
        for dc in range(-1, 2):
            if df == 0 and dc == 0:
                continue

            fila_v = f + df
            col_v = c + dc

            if (fila_v >= 0 and fila_v < ALTO) and (col_v >= 0 and col_v < ANCHO):
                if tablero[fila_v][col_v] == 1:
                    total += 1
    return total


def calcular_siguiente_generacion(tablero_base):

    siguiente_gen = []
    for f in range(ALTO):
        fila = []
        for c in range(ANCHO):
            fila.append(0)
        siguiente_gen.append(fila)

    for f in range(ALTO):
        for c in range(ANCHO):
            celda_actual = tablero_base[f][c]
            vecinos_vivos = contar_vecinos_vivos(tablero_base, f, c)

            if celda_actual == 1:
                if vecinos_vivos == 2 or vecinos_vivos == 3:
                    siguiente_gen[f][c] = 1
                else:
                    siguiente_gen[f][c] = 0
            else:
                if vecinos_vivos == 3:
                    siguiente_gen[f][c] = 1
                else:
                    siguiente_gen[f][c] = 0

    return siguiente_gen



for g in range(GENERACIONES_A_MOSTRAR):
    mostrar_tablero(tablero_actual, g + 1)

    tablero_actual = calcular_siguiente_generacion(tablero_actual)

    time.sleep(1.5)

### Explicación: Necesidad de un Nuevo Tablero en la Transición de Estados

* **Sincronía de los estados (Tiempo $T$ vs. Tiempo $T+1$):**
  En el Juego de la Vida de Conway, el estado de **todas** las celdas en la generación $T+1$ debe evaluarse de forma simultánea tomando como referencia única la fotografía del tablero en la generación $T$.

* **Problema de la modificación *in-place* (sobre el mismo tablero):**
  Si se actualizara directamente la matriz original mientras se recorre con los ciclos `for`:
  1. Las primeras celdas evaluadas cambiarían su valor de $1$ a $0$ (o viceversa) en la memoria RAM.
  2. Cuando las celdas contiguas o posteriores ejecuten la función `contar_vecinos_vivos()`, leerían el nuevo estado ($T+1$) de sus vecinas recién procesadas en lugar de su estado original ($T$).
  3. Esto generaría una cascada de datos corruptos, alterando las reglas lógicas y destruyendo el comportamiento del patrón (como el *Glider*).

---

### Trazabilidad de Memoria en el Algoritmo Imperativo

1. **Lectura estricta:** `calcular_siguiente_generacion()` consulta la matriz `tablero_base` únicamente en modo lectura para calcular `vecinos_vivos`.
2. **Aislamiento del nuevo estado:** Todos los resultados de la generación $T+1$ se escriben en una estructura matricial totalmente nueva denominada `siguiente_gen`.
3. **Reasignación del puntero:** Al finalizar el cálculo de la grilla completa, la variable `tablero_actual` se reasigna para apuntar a la dirección de memoria de `siguiente_gen`:

tablero_actual = calcular_siguiente_generacion(tablero_actual)